# 3. Vector Database 分片与副本：怎样 scatter-gather 全局 Top-K，并安全故障切换？

## 面试回答主线

向量库分片通常用稳定哈希把 doc_id 映射到 shard，再为每个 shard 维护一个或多个副本。查询不能只访问本地 shard，否则得到的是局部 top-k；正确流程是向所有相关 shard 并发 scatter，本地计算 cosine top-k，再由 coordinator 做全局 merge。副本用于可用性，但 failover 必须检查版本水位，否则主副延迟会返回过期内容。面试时我会用真实文档向量手写稳定分片、局部检索和全局 top-k，并让一个主副本故障以观察路由。结果表同时报告 baseline 局部命中与全局命中，而不是只打印向量形状。生产系统还要处理一致性级别、再平衡、热点、过滤条件和 tombstone。

## 1. 真实案例：十二篇知识文档与六个语义查询

六个主题分别是退款、天气、数据库、安全、RAG 和账单，每个主题有一篇精确文档与一篇相近干扰文档。六维向量具有可读主题坐标；doc_id 通过 SHA-256 稳定哈希映射到三个 shard。

In [1]:
from pprint import pprint  # 导入结构化打印工具展示分片、向量和检索结果
import hashlib  # 导入稳定哈希函数实现跨进程一致的 doc_id 分片
import numpy as np  # 导入 NumPy 手写余弦相似度与向量检索
documents = [{"id": "D01", "title": "未发货订单七天退款", "vector": [1.0, 0.0, 0.0, 0.0, 0.0, 0.0], "version": 1}, {"id": "D03", "title": "售后申请入口", "vector": [0.8, 0.1, 0.0, 0.0, 0.0, 0.0], "version": 1}, {"id": "D02", "title": "北京暴雨出行预警", "vector": [0.0, 1.0, 0.0, 0.0, 0.0, 0.0], "version": 1}, {"id": "D05", "title": "日常天气查询", "vector": [0.1, 0.8, 0.0, 0.0, 0.0, 0.0], "version": 1}, {"id": "D04", "title": "慢查询执行计划排查", "vector": [0.0, 0.0, 1.0, 0.0, 0.0, 0.0], "version": 1}, {"id": "D06", "title": "数据库连接池配置", "vector": [0.0, 0.1, 0.8, 0.0, 0.0, 0.0], "version": 1}, {"id": "D08", "title": "密钥泄露立即轮换", "vector": [0.0, 0.0, 0.0, 1.0, 0.0, 0.0], "version": 1}, {"id": "D07", "title": "普通账号登录帮助", "vector": [0.0, 0.0, 0.1, 0.8, 0.0, 0.0], "version": 1}, {"id": "D10", "title": "RAG 回答附来源引用", "vector": [0.0, 0.0, 0.0, 0.0, 1.0, 0.0], "version": 1}, {"id": "D09", "title": "向量索引参数说明", "vector": [0.0, 0.0, 0.0, 0.1, 0.8, 0.0], "version": 1}, {"id": "D11", "title": "企业账单导出流程", "vector": [0.0, 0.0, 0.0, 0.0, 0.0, 1.0], "version": 1}, {"id": "D12", "title": "发票抬头修改", "vector": [0.1, 0.0, 0.0, 0.0, 0.0, 0.8], "version": 1}]  # 定义十二篇带语义向量的真实文档
queries = [{"id": "V01", "text": "订单退款期限", "vector": [0.95, 0.05, 0.0, 0.0, 0.0, 0.0], "expected": "D01"}, {"id": "V02", "text": "北京暴雨要注意什么", "vector": [0.0, 0.95, 0.05, 0.0, 0.0, 0.0], "expected": "D02"}, {"id": "V03", "text": "慢查询怎么看执行计划", "vector": [0.0, 0.0, 1.0, 0.0, 0.0, 0.0], "expected": "D04"}, {"id": "V04", "text": "仓库密钥泄露怎么办", "vector": [0.0, 0.0, 0.0, 0.98, 0.02, 0.0], "expected": "D08"}, {"id": "V05", "text": "RAG 如何返回证据", "vector": [0.0, 0.0, 0.0, 0.0, 1.0, 0.0], "expected": "D10"}, {"id": "V06", "text": "企业账单怎样下载", "vector": [0.0, 0.0, 0.0, 0.0, 0.05, 0.95], "expected": "D11"}]  # 定义六个有明确目标文档的真实查询
shard_count = 3  # 设置三个逻辑向量分片
def shard_for(document_id):  # 用稳定哈希计算文档所属 shard
    digest = hashlib.sha256(document_id.encode("utf-8")).hexdigest()  # 生成与 Python 随机种子无关的文档哈希
    return int(digest[:8], 16) % shard_count  # 将哈希前缀映射到三个分片
shards = {shard_id: [document for document in documents if shard_for(document["id"]) == shard_id] for shard_id in range(shard_count)}  # 按稳定规则构造三个文档分片
preview = [{"doc_id": document["id"], "标题": document["title"], "shard": shard_for(document["id"]), "version": document["version"]} for document in documents]  # 汇总文档分片与版本信息
print("向量文档分片预览：")  # 输出真实案例标题
pprint(preview, sort_dicts=False)  # 展示十二篇文档在三个 shard 的真实分布

向量文档分片预览：
[{'doc_id': 'D01', '标题': '未发货订单七天退款', 'shard': 2, 'version': 1},
 {'doc_id': 'D03', '标题': '售后申请入口', 'shard': 1, 'version': 1},
 {'doc_id': 'D02', '标题': '北京暴雨出行预警', 'shard': 1, 'version': 1},
 {'doc_id': 'D05', '标题': '日常天气查询', 'shard': 0, 'version': 1},
 {'doc_id': 'D04', '标题': '慢查询执行计划排查', 'shard': 0, 'version': 1},
 {'doc_id': 'D06', '标题': '数据库连接池配置', 'shard': 0, 'version': 1},
 {'doc_id': 'D08', '标题': '密钥泄露立即轮换', 'shard': 2, 'version': 1},
 {'doc_id': 'D07', '标题': '普通账号登录帮助', 'shard': 0, 'version': 1},
 {'doc_id': 'D10', '标题': 'RAG 回答附来源引用', 'shard': 1, 'version': 1},
 {'doc_id': 'D09', '标题': '向量索引参数说明', 'shard': 0, 'version': 1},
 {'doc_id': 'D11', '标题': '企业账单导出流程', 'shard': 2, 'version': 1},
 {'doc_id': 'D12', '标题': '发票抬头修改', 'shard': 2, 'version': 1}]


## 2. Baseline（基线）：只查询 coordinator 本地 shard 0

局部 top-1 算法本身没有错，但它看不到其他两个 shard。下面手写 cosine，并让所有查询只访问 shard 0；返回结果可能相似，却不是全局最近文档。

In [2]:
def cosine(left, right):  # 手写两个稠密语义向量的余弦相似度
    left_array = np.asarray(left, dtype=float)  # 将查询向量转换为 NumPy 数组
    right_array = np.asarray(right, dtype=float)  # 将文档向量转换为 NumPy 数组
    denominator = np.linalg.norm(left_array) * np.linalg.norm(right_array)  # 计算两个向量范数乘积
    return float(np.dot(left_array, right_array) / denominator) if denominator else 0.0  # 返回归一化内积并处理零向量
def local_topk(query_vector, shard_documents, k):  # 在单个 shard 内执行精确 cosine top-k
    scored = [(document, cosine(query_vector, document["vector"])) for document in shard_documents]  # 计算查询与分片内每篇文档的相似度
    return sorted(scored, key=lambda pair: (-pair[1], pair[0]["id"]))[:k]  # 按分数降序和 doc_id 稳定排序
baseline_rows = []  # 收集只查本地 shard 的逐查询结果
for query in queries:  # 遍历六个真实语义查询
    local_results = local_topk(query["vector"], shards[0], k=1)  # 错误地只访问 coordinator 的本地分片
    document, score = local_results[0]  # 读取局部最高相似文档
    baseline_rows.append({"查询": query["id"], "问题": query["text"], "局部命中": document["id"], "标题": document["title"], "分数": round(score, 4), "全局目标": query["expected"], "正确": document["id"] == query["expected"]})  # 保存逐查询局部检索结果
print("只查 shard 0 的局部基线：")  # 标注当前输出属于错误查询范围基线
pprint(baseline_rows, sort_dicts=False)  # 展示局部 top-1 如何错过其他 shard 文档

只查 shard 0 的局部基线：
[{'查询': 'V01',
  '问题': '订单退款期限',
  '局部命中': 'D05',
  '标题': '日常天气查询',
  '分数': 0.176,
  '全局目标': 'D01',
  '正确': False},
 {'查询': 'V02',
  '问题': '北京暴雨要注意什么',
  '局部命中': 'D05',
  '标题': '日常天气查询',
  '分数': 0.9909,
  '全局目标': 'D02',
  '正确': False},
 {'查询': 'V03',
  '问题': '慢查询怎么看执行计划',
  '局部命中': 'D04',
  '标题': '慢查询执行计划排查',
  '分数': 1.0,
  '全局目标': 'D04',
  '正确': True},
 {'查询': 'V04',
  '问题': '仓库密钥泄露怎么办',
  '局部命中': 'D07',
  '标题': '普通账号登录帮助',
  '分数': 0.9921,
  '全局目标': 'D08',
  '正确': False},
 {'查询': 'V05',
  '问题': 'RAG 如何返回证据',
  '局部命中': 'D09',
  '标题': '向量索引参数说明',
  '分数': 0.9923,
  '全局目标': 'D10',
  '正确': False},
 {'查询': 'V06',
  '问题': '企业账单怎样下载',
  '局部命中': 'D09',
  '标题': '向量索引参数说明',
  '分数': 0.0522,
  '全局目标': 'D11',
  '正确': False}]


## 3. 核心机制：副本路由、scatter 与每 shard 局部 top-k

每个 shard 有 primary 与 replica 两个逻辑副本。为复现故障切换，我们把 shard 1 primary 标为不可用；路由器选择健康且版本水位足够的 replica。查询随后 scatter 到三个 shard，各自返回 top-2。

In [3]:
replicas = {(shard_id, "primary"): [dict(document) for document in shards[shard_id]] for shard_id in range(shard_count)}  # 为每个 shard 建立主副本数据视图
replicas.update({(shard_id, "replica"): [dict(document) for document in shards[shard_id]] for shard_id in range(shard_count)})  # 建立内容相同的第二副本
health = {(shard_id, role): True for shard_id in range(shard_count) for role in ("primary", "replica")}  # 初始化全部副本健康状态
health[(1, "primary")] = False  # 模拟 shard 1 主副本故障以触发真实 failover
def choose_replica(shard_id):  # 为一个逻辑 shard 选择健康读取副本
    for role in ("primary", "replica"):  # 优先尝试主副本再回退从副本
        if health[(shard_id, role)]:  # 检查当前副本是否可接受查询
            return role, replicas[(shard_id, role)]  # 返回副本角色和对应文档视图
    raise RuntimeError(f"shard_unavailable:{shard_id}")  # 全部副本失败时显式报告分片不可用
def scatter_query(query, local_k=2):  # 将一个向量查询分发到全部逻辑 shard
    shard_results = {}  # 收集每个 shard 的路由与局部 top-k
    for shard_id in range(shard_count):  # 遍历查询覆盖的三个逻辑分片
        role, replica_documents = choose_replica(shard_id)  # 选择当前可用且优先级最高的副本
        shard_results[shard_id] = {"role": role, "results": local_topk(query["vector"], replica_documents, local_k)}  # 在选定副本上计算局部相似度
    return shard_results  # 返回 coordinator 可合并的分片结果
sample_scatter = scatter_query(queries[1], local_k=2)  # 对北京暴雨查询执行一次完整 scatter
sample_ledger = [{"shard": shard_id, "读取副本": payload["role"], "局部top2": [(document["id"], round(score, 4)) for document, score in payload["results"]]} for shard_id, payload in sample_scatter.items()]  # 将每 shard 中间结果转为可读账本
print("北京暴雨查询的 scatter 轨迹：")  # 输出核心分片检索中间量标题
pprint(sample_ledger, sort_dicts=False)  # 展示主副路由与三个局部 top-2

北京暴雨查询的 scatter 轨迹：
[{'shard': 0, '读取副本': 'primary', '局部top2': [('D05', 0.9909), ('D06', 0.176)]},
 {'shard': 1, '读取副本': 'replica', '局部top2': [('D02', 0.9986), ('D03', 0.1239)]},
 {'shard': 2, '读取副本': 'primary', '局部top2': [('D01', 0.0), ('D08', 0.0)]}]


## 4. 手写全局 merge：对局部候选重新排序并去重

coordinator 汇总三个 shard 的候选，再按 cosine 分数做全局 top-k。因为稳定哈希保证每个 doc_id 只有一个逻辑 shard，正常情况下不会重复；副本只用于同 shard 内路由，不能同时返回两份。

In [4]:
def global_topk(query, k=1):  # 对全部 shard 的局部候选执行 coordinator 全局合并
    scattered = scatter_query(query, local_k=max(k, 2))  # 从每个逻辑 shard 取得足够的局部候选
    merged = []  # 初始化跨 shard 候选集合
    for shard_id, payload in scattered.items():  # 遍历三个 shard 的局部结果
        for document, score in payload["results"]:  # 展开当前 shard 的候选文档
            merged.append({"document": document, "score": score, "shard": shard_id, "replica": payload["role"]})  # 保留文档、分数和读取路由
    deduplicated = {}  # 以 doc_id 去重防止异常重复路由
    for candidate in merged:  # 逐个处理跨 shard 候选
        document_id = candidate["document"]["id"]  # 读取候选的稳定文档标识
        if document_id not in deduplicated or candidate["score"] > deduplicated[document_id]["score"]:  # 保留同文档最高分副本
            deduplicated[document_id] = candidate  # 更新全局候选索引
    ranked = sorted(deduplicated.values(), key=lambda item: (-item["score"], item["document"]["id"]))  # 对去重候选执行稳定全局排序
    return ranked[:k]  # 返回真正跨全部 shard 的 top-k
global_rows = []  # 收集六个查询的全局检索结果
for query in queries:  # 遍历相同查询集合
    best = global_topk(query, k=1)[0]  # scatter 三个 shard 并取得全局 top-1
    global_rows.append({"查询": query["id"], "问题": query["text"], "全局命中": best["document"]["id"], "标题": best["document"]["title"], "分数": round(best["score"], 4), "来自shard": best["shard"], "读取副本": best["replica"], "正确": best["document"]["id"] == query["expected"]})  # 保存逐查询全局结果与路由证据
print("scatter-gather 全局 Top-1：")  # 输出核心方案结果标题
pprint(global_rows, sort_dicts=False)  # 展示六个主题的正确文档和所属分片

scatter-gather 全局 Top-1：
[{'查询': 'V01',
  '问题': '订单退款期限',
  '全局命中': 'D01',
  '标题': '未发货订单七天退款',
  '分数': 0.9986,
  '来自shard': 2,
  '读取副本': 'primary',
  '正确': True},
 {'查询': 'V02',
  '问题': '北京暴雨要注意什么',
  '全局命中': 'D02',
  '标题': '北京暴雨出行预警',
  '分数': 0.9986,
  '来自shard': 1,
  '读取副本': 'replica',
  '正确': True},
 {'查询': 'V03',
  '问题': '慢查询怎么看执行计划',
  '全局命中': 'D04',
  '标题': '慢查询执行计划排查',
  '分数': 1.0,
  '来自shard': 0,
  '读取副本': 'primary',
  '正确': True},
 {'查询': 'V04',
  '问题': '仓库密钥泄露怎么办',
  '全局命中': 'D08',
  '标题': '密钥泄露立即轮换',
  '分数': 0.9998,
  '来自shard': 2,
  '读取副本': 'primary',
  '正确': True},
 {'查询': 'V05',
  '问题': 'RAG 如何返回证据',
  '全局命中': 'D10',
  '标题': 'RAG 回答附来源引用',
  '分数': 1.0,
  '来自shard': 1,
  '读取副本': 'replica',
  '正确': True},
 {'查询': 'V06',
  '问题': '企业账单怎样下载',
  '全局命中': 'D11',
  '标题': '企业账单导出流程',
  '分数': 0.9986,
  '来自shard': 2,
  '读取副本': 'primary',
  '正确': True}]


## 5. 结果解读：局部精确不等于全局召回

局部基线只有目标恰好位于 shard 0 时才能命中；全局方案在同样 cosine 和向量上恢复六个目标。副本 failover 不改变逻辑搜索范围，也不应把主副同时当成两个 shard。

In [5]:
baseline_hits = sum(row["正确"] for row in baseline_rows)  # 统计只查 shard 0 的目标命中数量
global_hits = sum(row["正确"] for row in global_rows)  # 统计 scatter-gather 的目标命中数量
comparison = [{"查询": query["id"], "目标": query["expected"], "局部shard0": baseline_rows[index]["局部命中"], "全局top1": global_rows[index]["全局命中"], "全局读取路由": f'{global_rows[index]["来自shard"]}/{global_rows[index]["读取副本"]}'} for index, query in enumerate(queries)]  # 构造同数据逐查询对照
print("局部与全局检索对照：")  # 输出结果解读标题
pprint(comparison, sort_dicts=False)  # 展示局部范围限制而非 cosine 算法错误
print({"局部命中": f"{baseline_hits}/{len(queries)}", "全局命中": f"{global_hits}/{len(queries)}", "shard1发生主副切换": any(row["来自shard"] == 1 and row["读取副本"] == "replica" for row in global_rows)})  # 汇总召回与可用性结果

局部与全局检索对照：
[{'查询': 'V01',
  '目标': 'D01',
  '局部shard0': 'D05',
  '全局top1': 'D01',
  '全局读取路由': '2/primary'},
 {'查询': 'V02',
  '目标': 'D02',
  '局部shard0': 'D05',
  '全局top1': 'D02',
  '全局读取路由': '1/replica'},
 {'查询': 'V03',
  '目标': 'D04',
  '局部shard0': 'D04',
  '全局top1': 'D04',
  '全局读取路由': '0/primary'},
 {'查询': 'V04',
  '目标': 'D08',
  '局部shard0': 'D07',
  '全局top1': 'D08',
  '全局读取路由': '2/primary'},
 {'查询': 'V05',
  '目标': 'D10',
  '局部shard0': 'D09',
  '全局top1': 'D10',
  '全局读取路由': '1/replica'},
 {'查询': 'V06',
  '目标': 'D11',
  '局部shard0': 'D09',
  '全局top1': 'D11',
  '全局读取路由': '2/primary'}]
{'局部命中': '1/6', '全局命中': '6/6', 'shard1发生主副切换': True}


## 6. 失败案例与修正：主副切换读到低版本文档

假设退款政策 D01 已在 primary 更新到 v2“3 天内”，replica 仍停在 v1“7 天内”。主副本故障时，朴素 failover 会返回语义最相近但内容过期的 v1。修正是请求携带 required_version 或 consistency watermark，副本水位不足则拒绝读并触发 read-repair。

In [6]:
primary_record = {"id": "D01", "version": 2, "text": "未发货订单 3 天内可退"}  # 构造已提交到主副本的最新退款政策
lagging_record = {"id": "D01", "version": 1, "text": "未发货订单 7 天内可退"}  # 构造尚未追上主副本的旧副本内容
required_version = 2  # 模拟查询端从元数据服务获得的最低可接受版本
naive_failover_answer = lagging_record["text"]  # 复现主副本故障后无版本检查直接返回旧答案
replica_is_fresh = lagging_record["version"] >= required_version  # 比较副本水位与读取一致性要求
safe_decision = lagging_record["text"] if replica_is_fresh else "REJECT_STALE_AND_READ_REPAIR"  # 低版本副本不允许静默提供答案
repaired_record = dict(primary_record)  # 模拟从变更日志或其他健康副本执行 read-repair
print({"失败_朴素failover": naive_failover_answer, "副本版本": lagging_record["version"], "要求版本": required_version, "安全决策": safe_decision, "修复后记录": repaired_record})  # 展示可用性与一致性冲突及修正

{'失败_朴素failover': '未发货订单 7 天内可退', '副本版本': 1, '要求版本': 2, '安全决策': 'REJECT_STALE_AND_READ_REPAIR', '修复后记录': {'id': 'D01', 'version': 2, 'text': '未发货订单 3 天内可退'}}


## 7. 生产差距与最小回归检查

生产向量库还要处理 HNSW/IVF 近似召回、metadata filter、分片并发超时、热点副本与在线再平衡。写入路径需要 WAL、版本水位、tombstone 和 read-repair；查询可按业务选择 eventual、session 或 quorum consistency。最后的断言只验证本实验的稳定分片、局部漏召、全局 merge、failover 与版本门禁。

In [7]:
assert len(documents) >= 6 and len(queries) >= 6  # 确认真实文档与查询数量满足逐样本教学要求
assert set(shards) == {0, 1, 2} and sum(len(items) for items in shards.values()) == len(documents)  # 确认稳定哈希无遗漏地分布全部文档
assert baseline_hits < global_hits  # 确认只查局部 shard 的召回确实低于全局 scatter-gather
assert global_hits == len(queries)  # 确认六个真实查询都找回全局最近目标文档
assert health[(1, "primary")] is False and choose_replica(1)[0] == "replica"  # 确认 shard 1 主副故障切换真实发生
assert replica_is_fresh is False and safe_decision == "REJECT_STALE_AND_READ_REPAIR"  # 确认低版本副本不会静默返回旧政策
print("回归检查通过：稳定分片、全局 Top-K、副本故障切换与版本水位均已验证。")  # 输出最终验收结论

回归检查通过：稳定分片、全局 Top-K、副本故障切换与版本水位均已验证。
